# Optimisation: practice notebook

2 exercises, each with hidden tests. Data is loaded as **polars**; pandas only appears where a library insists on it.

**How to use it**
1. Run the three setup cells below once (Runtime → Run all works too; the exercise cells just print `Wrong answer` until you solve them).
2. For each exercise, write your code in the solution cell, run it, then run the `check(...)` cell under it. It prints **Accepted** or which test failed and why.
3. Stuck? `solution("o1")` prints the reference solution.

Opened from the roadmap it runs in Google Colab. To keep your work, use File → Save a copy in Drive the first time.

In [ ]:
# Install anything missing (Colab already has numpy, scipy, statsmodels, scikit-learn, matplotlib)
import importlib.util, subprocess, sys
for _p in ['polars']:
    if importlib.util.find_spec(_p) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p], check=False)


## Setup: data
The data every exercise uses. Read it if you like; don't change it.

In [ ]:
import numpy as np
from scipy.optimize import minimize
mu = np.array([0.06, 0.08, 0.10, 0.04, 0.12])
vol = np.array([0.15, 0.20, 0.25, 0.08, 0.30])
corr = np.full((5, 5), 0.3) + np.eye(5) * 0.7
cov = np.outer(vol, vol) * corr

In [ ]:
# Test runner: check(id) and solution(id)
import base64 as _b64, json as _json, io as _io, contextlib as _ctx, traceback as _tb
_LAB = _json.loads(_b64.b64decode("eyJvMSI6IHsic29sdXRpb24iOiAiZGVmIGxhZ3JhbmdlX3FwKFEsIGIsIGEpOlxuICAgIG4gPSBsZW4oYilcbiAgICBLID0gbnAuYmxvY2soW1tRLCBhLnJlc2hhcGUoLTEsIDEpXSwgW2EucmVzaGFwZSgxLCAtMSksIG5wLnplcm9zKCgxLCAxKSldXSlcbiAgICByZXR1cm4gbnAubGluYWxnLnNvbHZlKEssIG5wLmFwcGVuZChiLCAxLjApKVs6bl1cblxuZGVmIHNjaXB5X3FwKFEsIGIsIGEpOlxuICAgIGYgPSBsYW1iZGEgeDogMC41ICogeCBAIFEgQCB4IC0gYiBAIHhcbiAgICBjb25zID0gW3tcInR5cGVcIjogXCJlcVwiLCBcImZ1blwiOiBsYW1iZGEgeDogYSBAIHggLSAxfV1cbiAgICByZXR1cm4gbWluaW1pemUoZiwgbnAuemVyb3MobGVuKGIpKSwgamFjPWxhbWJkYSB4OiBRIEAgeCAtIGIsIGNvbnN0cmFpbnRzPWNvbnMsIG1ldGhvZD1cIlNMU1FQXCIsIG9wdGlvbnM9e1wiZnRvbFwiOiAxZS0xMn0pLnhcbiIsICJ0ZXN0cyI6ICJcbmRlZiBjbG9zZShhLCBiLCByZWw9MWUtNiwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYSA9IGZsb2F0KGEpOyBiID0gZmxvYXQoYilcbiAgICBhc3NlcnQgYWJzKGEgLSBiKSA8PSByZWwgKiBtYXgoMS4wLCBhYnMoYikpLCBmXCJ7d2hhdH0gPSB7YTouNmd9LCBleHBlY3RlZCB7YjouNmd9XCJcbmRlZiBiZXR3ZWVuKGEsIGxvLCBoaSwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYXNzZXJ0IGxvIDw9IGZsb2F0KGEpIDw9IGhpLCBmXCJ7d2hhdH0gPSB7ZmxvYXQoYSk6LjRmfSwgZXhwZWN0ZWQgYmV0d2VlbiB7bG99IGFuZCB7aGl9XCJcbmRlZiBzYW1lX2tleXMoYSwgYiwgd2hhdD1cInJlc3VsdFwiKTpcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShhLCBkaWN0KSwgZlwie3doYXR9IHNob3VsZCBiZSBhIGRpY3QsIGdvdCB7dHlwZShhKS5fX25hbWVfX31cIlxuICAgIGFzc2VydCBzZXQoYSkgPT0gc2V0KGIpLCBmXCJ7d2hhdH0ga2V5cyB7c29ydGVkKGEpfSwgZXhwZWN0ZWQge3NvcnRlZChiKX1cIlxuXG5kZWYgX3QoUSwgYiwgYSk6XG4gICAgeCA9IG5wLmFzYXJyYXkoUltcImxhZ3JhbmdlX3FwXCJdKFEsIGIsIGEpKVxuICAgIGZvciBuYW1lLCB0b2wgaW4gKChcImxhZ3JhbmdlX3FwXCIsIDFlLTgpLCAoXCJzY2lweV9xcFwiLCAxZS00KSk6XG4gICAgICAgIHkgPSBucC5hc2FycmF5KFVbbmFtZV0oUSwgYiwgYSkpXG4gICAgICAgIGFzc2VydCB5LnNoYXBlID09IHguc2hhcGUsIGZcIntuYW1lfTogc2hhcGUge3kuc2hhcGV9LCBleHBlY3RlZCB7eC5zaGFwZX1cIlxuICAgICAgICBmb3IgaSBpbiByYW5nZShsZW4oeCkpOiBjbG9zZSh5W2ldLCB4W2ldLCB0b2wsIGZcIntuYW1lfSB4W3tpfV1cIilcblRFU1RTID0gWyhcImRpYWdvbmFsIFFcIiwgbGFtYmRhOiBfdChucC5kaWFnKFsyLjAsIDQuMCwgNi4wXSksIG5wLmFycmF5KFsxLjAsIDAuMCwgMS4wXSksIG5wLm9uZXMoMykpKSxcbiAgICAgICAgIChcImhpZGRlbjogY29ycmVsYXRlZCBRXCIsIGxhbWJkYTogX3QobnAuYXJyYXkoW1s0LCAxLCAwLjVdLCBbMSwgMywgMC4yXSwgWzAuNSwgMC4yLCAyLjBdXSksIG5wLmFycmF5KFswLjUsIC0xLCAyLjBdKSwgbnAuYXJyYXkoWzEuMCwgMi4wLCAtMS4wXSkpKV1cbiIsICJsYW5nIjogInB5In0sICJvMiI6IHsic29sdXRpb24iOiAiZGVmIG1heF9zaGFycGUobXUsIGNvdiwgcmY9MC4wMik6XG4gICAgbiA9IGxlbihtdSlcbiAgICBuZWcgPSBsYW1iZGEgdzogLSh3IEAgbXUgLSByZikgLyBucC5zcXJ0KHcgQCBjb3YgQCB3KVxuICAgIGNvbnMgPSBbe1widHlwZVwiOiBcImVxXCIsIFwiZnVuXCI6IGxhbWJkYSB3OiB3LnN1bSgpIC0gMX1dXG4gICAgcmV0dXJuIG1pbmltaXplKG5lZywgbnAuZnVsbChuLCAxIC8gbiksIGJvdW5kcz1bKDAsIDEpXSAqIG4sIGNvbnN0cmFpbnRzPWNvbnMsIG1ldGhvZD1cIlNMU1FQXCIsIG9wdGlvbnM9e1wiZnRvbFwiOiAxZS0xMn0pLnhcbiIsICJ0ZXN0cyI6ICJcbmRlZiBjbG9zZShhLCBiLCByZWw9MWUtNiwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYSA9IGZsb2F0KGEpOyBiID0gZmxvYXQoYilcbiAgICBhc3NlcnQgYWJzKGEgLSBiKSA8PSByZWwgKiBtYXgoMS4wLCBhYnMoYikpLCBmXCJ7d2hhdH0gPSB7YTouNmd9LCBleHBlY3RlZCB7YjouNmd9XCJcbmRlZiBiZXR3ZWVuKGEsIGxvLCBoaSwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYXNzZXJ0IGxvIDw9IGZsb2F0KGEpIDw9IGhpLCBmXCJ7d2hhdH0gPSB7ZmxvYXQoYSk6LjRmfSwgZXhwZWN0ZWQgYmV0d2VlbiB7bG99IGFuZCB7aGl9XCJcbmRlZiBzYW1lX2tleXMoYSwgYiwgd2hhdD1cInJlc3VsdFwiKTpcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShhLCBkaWN0KSwgZlwie3doYXR9IHNob3VsZCBiZSBhIGRpY3QsIGdvdCB7dHlwZShhKS5fX25hbWVfX31cIlxuICAgIGFzc2VydCBzZXQoYSkgPT0gc2V0KGIpLCBmXCJ7d2hhdH0ga2V5cyB7c29ydGVkKGEpfSwgZXhwZWN0ZWQge3NvcnRlZChiKX1cIlxuXG5kZWYgX3QobSwgYywgcmYpOlxuICAgIHcgPSBucC5hc2FycmF5KFVbXCJtYXhfc2hhcnBlXCJdKG0sIGMsIHJmKSk7IHIgPSBucC5hc2FycmF5KFJbXCJtYXhfc2hhcnBlXCJdKG0sIGMsIHJmKSlcbiAgICBjbG9zZSh3LnN1bSgpLCAxLjAsIDFlLTYsIFwic3VtIG9mIHdlaWdodHNcIilcbiAgICBhc3NlcnQgdy5taW4oKSA+PSAtMWUtNiwgZlwiYSB3ZWlnaHQgaXMgbmVnYXRpdmU6IHt3Lm1pbigpOi40Z31cIlxuICAgIHNoID0gbGFtYmRhIHY6ICh2IEAgbSAtIHJmKSAvIG5wLnNxcnQodiBAIGMgQCB2KVxuICAgIGNsb3NlKHNoKHcpLCBzaChyKSwgMWUtNSwgXCJTaGFycGUgcmF0aW9cIilcblRFU1RTID0gWyhcImZpdmUgYXNzZXRzXCIsIGxhbWJkYTogX3QobXUsIGNvdiwgMC4wMikpLFxuICAgICAgICAgKFwiaGlkZGVuOiBhIGRvbWluYXRlZCBhc3NldCBtdXN0IGdldCAwXCIsIGxhbWJkYTogX3QobnAuYXJyYXkoWzAuMDYsIDAuMDgsIDAuMDIsIDAuMTBdKSwgY292Wzo0LCA6NF0sIDAuMDEpKV1cbiIsICJsYW5nIjogInB5In19").decode())
_SETUP = _b64.b64decode("aW1wb3J0IG51bXB5IGFzIG5wCmZyb20gc2NpcHkub3B0aW1pemUgaW1wb3J0IG1pbmltaXplCm11ID0gbnAuYXJyYXkoWzAuMDYsIDAuMDgsIDAuMTAsIDAuMDQsIDAuMTJdKQp2b2wgPSBucC5hcnJheShbMC4xNSwgMC4yMCwgMC4yNSwgMC4wOCwgMC4zMF0pCmNvcnIgPSBucC5mdWxsKCg1LCA1KSwgMC4zKSArIG5wLmV5ZSg1KSAqIDAuNwpjb3YgPSBucC5vdXRlcih2b2wsIHZvbCkgKiBjb3Jy").decode()

def _err(e):
    fr = [f for f in _tb.extract_tb(e.__traceback__) if f.filename.startswith("<ipython") or f.filename == "<string>"]
    return f"{type(e).__name__}: {e}"

def check(ex_id, query=None):
    """Run the hidden tests for one exercise, like Submit in the editor."""
    e = _LAB[ex_id]
    ref = {}
    with _ctx.redirect_stdout(_io.StringIO()):
        exec(_SETUP, ref)
        if e["lang"] == "sql":
            ref["__sql__"] = e["solution"]
        else:
            exec(e["solution"], ref)
    if e["lang"] == "sql":
        if query is None:
            print("Pass your query string: check(\"" + ex_id + "\", q)")
            return
        user = {"__sql__": query}
    else:
        user = dict(globals())
    tns = dict(ref)
    tns["U"], tns["R"] = user, ref
    exec(e["tests"], tns)
    ok, tests = 0, tns["TESTS"]
    for name, fn in tests:
        try:
            fn()
            print("  ✓ " + name)
            ok += 1
        except AssertionError as err:
            print("  ✗ " + name + ": " + (str(err) or "wrong answer"))
        except Exception as err:
            print("  ✗ " + name + ": " + _err(err))
    print(("Accepted" if ok == len(tests) else "Wrong answer") + f" · {ok} / {len(tests)} test cases passed")

def solution(ex_id):
    """Print the reference solution. Try once more first."""
    print(_LAB[ex_id]["solution"])

---
## 1. scipy.optimize.minimize: a quadratic with an equality constraint, checked against your Lagrange solution
*Easy · id `o1`*

<p>Minimise f(x) = ½ xᵀQx − bᵀx subject to aᵀx = 1, with Q positive definite.</p>
<p>1. On paper: write the Lagrangian and show the solution solves the linear system [[Q, a], [aᵀ, 0]] [x, λ] = [b, 1].</p>
<p>2. <code>lagrange_qp(Q, b, a)</code>: solve that system with <code>np.linalg.solve</code> and return x.</p>
<p>3. <code>scipy_qp(Q, b, a)</code>: the same with <code>minimize(..., method="SLSQP", constraints=[{"type": "eq", ...}])</code> starting from zeros. The two should agree.</p>

**Theory to know cold**
<ul><li>At the optimum ∇f = λ∇g: the gradient is perpendicular to the constraint surface.</li><li>λ is the shadow price: how much the optimal value changes if the constraint's right-hand side moves by 1.</li></ul>

In [ ]:
def lagrange_qp(Q, b, a):
    # your code here
    pass

def scipy_qp(Q, b, a):
    # your code here
    pass

# Sample run
try:
    Q = np.diag([2.0, 4.0, 6.0]); b = np.array([1.0, 0.0, 1.0]); a = np.ones(3)
    print(lagrange_qp(Q, b, a), scipy_qp(Q, b, a))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("o1")

---
## 2. scipy.optimize.minimize: long-only max-Sharpe portfolio with weights summing to 1
*Medium · id `o2`*

<p>Write <code>max_sharpe(mu, cov, rf=0.02)</code>: maximise (wᵀμ − r_f) / √(wᵀΣw) subject to Σwᵢ = 1 and wᵢ ≥ 0, with SLSQP (minimise the negative Sharpe, bounds (0, 1), start from equal weights). Return the weights.</p>
<p>Then compare with the unconstrained tangency portfolio w ∝ Σ⁻¹(μ − r_f): when do they coincide?</p>

**Theory to know cold**
<ul><li>Tangency weights ∝ Σ⁻¹(μ − r_f), normalised to sum to 1. If all are positive, the long-only constraint doesn't bind.</li><li>Mean-variance weights are very sensitive to μ: small changes in expected returns swing the weights, which is why practitioners shrink μ or Σ.</li></ul>

In [ ]:
def max_sharpe(mu, cov, rf=0.02):
    # your code here
    pass

# Sample run
try:
    w = max_sharpe(mu, cov); print(w.round(4), (w @ mu - 0.02) / np.sqrt(w @ cov @ w))
    t = np.linalg.solve(cov, mu - 0.02); print((t / t.sum()).round(4))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("o2")